# Stage D (Part 2) — Dynamic ERA5 **Daily** Features & Train Sets

**Purpose**
Engineer **daily** meteorological features from ERA5 at the basin level, join **static basin attributes**, and assemble **training datasets** for the 3 modeled basins.

---

## Inputs

* Static attributes (from Part 1): `data/modeling/static/basin_attributes.parquet`
* Station→basin selections (sanity): `data/modeling/targets/meta/station_to_basin_name.csv` + `data/boundaries/processed/basin_lookup.csv`

---

## What this notebook does

1. **Resolve project root** and set paths/knobs (windows, API half-lives, ARX lags).


3. **Variable catalog & base derivations**

   * Classify vars as **flux-like** (e.g., precipitation, runoff, snowmelt, radiation, PET) vs **state-like** (e.g., temperature, dewpoint, soil T, snow depth).
   * Derive **wind\_speed** from U/V; derive **VPD** from temperature & dewpoint.
4. **Leak-safe feature engineering (daily)**

   * **Flux windows (sums):** 1, 3, 7, 14, 30 days → `*_sum_{w}d` (uses **past** days only).
   * **State windows (means):** 1, 3, 7, 14 days → `*_mean_{w}d` (past days only).
   * **API (Antecedent Precipitation Index):** exponential memory of precip with **3** and **7** day half-lives → `api_d3`, `api_d7` (shifted to avoid leakage).
   * **Calendar/condition flags:** `month`, `doy_sin`, `doy_cos`, `is_monsoon` (Jun–Sep), `is_freezing` (T ≤ 0 °C), `has_snowpack` (snow\_depth > 0).
5. **Join statics**: merge `basin_attributes.parquet` onto each `(basin_id, date_local)` row.
6. **Save features-only table** → `data/modeling/features/era5_features_basin_daily.parquet`.
7. **Build training datasets**

   * **Exogenous-only:** features ⊕ target (`discharge_cms`, `qc_any`) → `data/modeling/datasets/train_basin_daily_exogenous.parquet`.
   * **ARX (optional):** add discharge lags `q_lag_{1,2,3,7,14,30}d` and rolling stats (`q_roll7_mean`, `q_roll14_std`); drop rows without full lag history → `data/modeling/datasets/train_basin_daily_arx.parquet`.

---

## Assumptions & conventions

* **Cadence:** daily only (no 6-hour expansion).
* **Time zone:** Bhutan local; `date_local` normalized to naive midnight.
* **Leak safety:** all rolling/lag features use **shifted** history (no future info).
* **Units:** ERA5 kept in native units; if precip is meters/day you can later add a `_mm` copy for readability.
* **QC:** `qc_any` from targets is carried through; no filtering/imputation applied here.

---

## QA printed by the notebook

* Modeled `basin_id`s; row counts before/after subsetting.
* Date window used (start → end).
* Detected flux/state variables; confirmation of derived `wind_speed`/`vpd_kpa`.
* Feature NA diagnostics: `feature_na_count` / `feature_na_frac`.
* Train set sizes; ARX rows dropped due to lag history.

---

## Outputs

* **Features (daily):** `data/modeling/features/era5_features_basin_daily.parquet`
* **Train — exogenous:** `data/modeling/datasets/train_basin_daily_exogenous.parquet`
* **Train — ARX (optional):** `data/modeling/datasets/train_basin_daily_arx.parquet`

---

## Tunable knobs (set at top of notebook)

* Flux windows: `[1, 3, 7, 14, 30]`
* State windows: `[1, 3, 7, 14]`
* API half-lives: `[3, 7]` days
* ARX lags: `[1, 2, 3, 7, 14, 30]` and rolling stats: `(7, mean)`, `(14, std)`
* Optional `ERA5_KEEP` whitelist to slim variables before engineering.


In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# === Resolve Project Root ===

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


In [3]:
# === Paths & Settings ===

# Inputs
FEATURES_DAILY_PARQUET = PROJECT_ROOT / "data/modeling/targets/train_basin_daily.parquet"
STATIC_ATTR_PARQUET   = PROJECT_ROOT / "data/modeling/static/basin_attributes.parquet"

In [14]:
# Outputs

TRAIN_EXOG_PARQUET     = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_exogenous.parquet"
TRAIN_ARX_PARQUET      = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_arx.parquet"

TRAIN_EXOG_PARQUET.parent.mkdir(parents=True, exist_ok=True)

In [5]:
# Feature knobs (daily cadence)
FLUX_WINDOWS  = [1, 3, 7, 14, 30]      # sums of past N full days
STATE_WINDOWS = [1, 3, 7, 14]          # means of past N full days
API_HALFLIFE  = [3, 7]                 # days (for antecedent precip index)
ARX_LAGS      = [1, 2, 3, 7, 14, 30]   # discharge lags (days)
ROLL_Q_STATS  = [(7, "mean"), (14, "std")]  # discharge rolling stats
PRODUCE_ARX   = True                   # also build ARX dataset (exogenous + lagged Q)

# Optional whitelist to slim ERA5 before engineering (None = keep all)
ERA5_KEEP = None

## Variable Catalog & Base Derivations (wind, VPD)

This block classifies the numeric ERA5 variables into **flux-like** (e.g., precipitation, runoff, radiation) and **state-like** (e.g., temperature, soil moisture) features for modeling.  
It also derives additional useful features:  
* **wind_speed** from available U/V wind components  
* **vpd_kpa** (vapor pressure deficit) from temperature and dewpoint, auto-detecting Kelvin vs °C  

Finally, it prints sample lists of flux and state variables, helping verify which features will be used downstream.


In [15]:
# Load features + targets
features = pd.read_parquet(FEATURES_DAILY_PARQUET)

# --- Pick numeric candidate columns (exclude IDs / time) ---
id_like = {"basin_id", "basin_name", "date_local"}
num_cols = [c for c in features.columns
            if c not in id_like and pd.api.types.is_numeric_dtype(features[c])]

# --- Heuristic classification: flux vs state ---
def classify_flux_state(cols):
    flux_like, state_like = set(), set()
    for c in cols:
        lc = c.lower()
        if any(k in lc for k in ["precip", "runoff", "snowmelt", "evap", "radiation", "ssrd"]):
            flux_like.add(c)
        else:
            state_like.add(c)
    return sorted(flux_like), sorted(state_like)

flux_vars, state_vars = classify_flux_state(num_cols)

In [16]:
# --- Derive wind_speed if U/V present (kept as state-like) ---
def add_wind_speed(df, state_vars):
    candidates = [
        ("u10", "v10"),
        ("u_component_of_wind_10m", "v_component_of_wind_10m"),
        ("wind_u", "wind_v"), ("u", "v"),
    ]
    for u_name, v_name in candidates:
        u = next((c for c in df.columns if u_name == c or u_name in c.lower()), None)
        v = next((c for c in df.columns if v_name == c or v_name in c.lower()), None)
        if u and v:
            if "wind_speed" not in df.columns:
                df["wind_speed"] = (df[u]**2 + df[v]**2) ** 0.5
            if "wind_speed" not in state_vars:
                state_vars = state_vars + ["wind_speed"]
            break
    if "wind_speed" in df.columns and "wind_speed" not in state_vars:
        state_vars = state_vars + ["wind_speed"]
    return df, state_vars

features, state_vars = add_wind_speed(features, state_vars)

In [17]:
# --- Derive VPD (auto-detect Kelvin vs °C) ---
def to_celsius(series: pd.Series) -> pd.Series:
    med = series.median(skipna=True)
    return series - 273.15 if pd.notna(med) and med > 200 else series

def find_col(df, name_hints):
    for h in name_hints:
        c = next((c for c in df.columns if h == c or h in c.lower()), None)
        if c: return c
    return None

temp_col = find_col(features, ["t2m","temperature","2m_temperature"])
dew_col  = find_col(features, ["d2m","dewpoint","dew_point","2m_dewpoint"])

if temp_col and dew_col and "vpd_kpa" not in features.columns:
    T_c  = to_celsius(features[temp_col])
    Td_c = to_celsius(features[dew_col])
    es = 0.6108 * np.exp(17.27 * T_c  / (T_c  + 237.3))
    ea = 0.6108 * np.exp(17.27 * Td_c / (Td_c + 237.3))
    features["vpd_kpa"] = (es - ea).clip(lower=0)
    if "vpd_kpa" not in state_vars:
        state_vars.append("vpd_kpa")

print("Flux-like vars (sample):", flux_vars[:12], "…")
print("State-like vars (incl. derived, sample):", [v for v in state_vars if v not in flux_vars][:12], "…")

Flux-like vars (sample): ['potential_evaporation', 'precipitation', 'runoff', 'snowmelt', 'solar_radiation', 'sub_surface_runoff', 'surface_runoff'] …
State-like vars (incl. derived, sample): ['dewpoint', 'discharge_cms', 'low_coverage', 'n_cells', 'qc_any', 'snow_depth', 'soil_temperature', 'temperature', 'wind_u', 'wind_v', 'wind_speed', 'vpd_kpa'] …


## Rolling & Lag Functions (Leak-safe)

This block defines **leak-safe rolling and lag functions** for time-series features, ensuring that only information from past days is used (no look-ahead bias).  

* `strict_sum_past` – rolling sum over the previous *N* full days, excluding today.  
* `strict_mean_past` – rolling mean over the previous *N* full days, excluding today.  
* `api_series` – computes an **Antecedent Precipitation Index (API)** or similar exponentially decaying memory of past fluxes, shifted by one day so that only data up to *t−1* is used.


In [7]:

# === Rolling & lag helpers (add once, then use as needed) ===
def strict_sum_past(s: pd.Series, window: int) -> pd.Series:
    """Rolling sum over the previous N full days, excluding today."""
    return s.shift(1).rolling(window=window, min_periods=window).sum()

def strict_mean_past(s: pd.Series, window: int) -> pd.Series:
    """Rolling mean over the previous N full days, excluding today."""
    return s.shift(1).rolling(window=window, min_periods=window).mean()

def api_series(s: pd.Series, half_life_days: int) -> pd.Series:
    """Antecedent Precipitation Index (API) using exponential decay, shifted to avoid look-ahead leakage."""
    k = 0.5 ** (1.0 / half_life_days)
    out = np.empty(len(s))
    out[:] = np.nan
    acc = 0.0
    for i, val in enumerate(s.fillna(0.0).values):
        acc = val + k * acc
        out[i] = acc
    return pd.Series(out, index=s.index).shift(1)


In [8]:
features

,basin_id,basin_name,date_local,discharge_cms,qc_any,temperature,dewpoint,wind_u,wind_v,potential_evaporation,...,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation,n_cells,low_coverage,wind_speed,vpd_kpa
0,3.0,Mangdechhu,2000-01-01,14.783000,False,-2.497898,265.813408,0.095126,-0.488079,-0.000222,...,0.000080,274.263428,0.000114,0.000029,2.368407e+06,0.000018,11,False,0.497263,0.156228
1,3.0,Mangdechhu,2000-01-02,14.620000,False,-2.066571,265.773514,0.254713,-0.419430,-0.000248,...,0.000075,274.279602,0.000114,0.000031,2.494330e+06,0.000029,11,False,0.490714,0.173844
2,3.0,Mangdechhu,2000-01-03,14.565000,False,-3.107548,266.192237,0.277529,-0.309926,-0.000225,...,0.000052,274.151822,0.000113,0.000028,2.318063e+06,0.000090,11,False,0.416025,0.123179
3,3.0,Mangdechhu,2000-01-04,14.031000,False,-3.349576,265.113591,0.199192,-0.453193,-0.000241,...,0.000042,274.176020,0.000112,0.000017,2.486289e+06,0.000015,11,False,0.495037,0.143576
4,3.0,Mangdechhu,2000-01-05,14.032000,False,-2.330868,266.624595,0.226005,-0.230016,-0.000214,...,0.000027,274.346547,0.000110,0.000011,2.144902e+06,0.000050,11,False,0.322468,0.139814
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29835,8.0,Punatsangchhu,2023-12-27,74.138000,False,1.583485,269.573770,-0.109918,0.135857,-0.000223,...,0.000015,275.859574,0.000124,0.000043,2.059552e+06,0.000339,14,False,0.174754,0.215921
29836,8.0,Punatsangchhu,2023-12-28,74.247002,False,1.807874,270.468924,0.019553,0.058206,-0.000246,...,0.000034,276.004281,0.000122,0.000058,2.135026e+06,0.000329,14,False,0.061403,0.194590
29837,8.0,Punatsangchhu,2023-12-29,74.027000,False,1.401505,268.092982,0.020120,-0.254086,-0.000276,...,0.000035,275.692326,0.000121,0.000023,2.384882e+06,0.000069,14,False,0.254881,0.256627
29838,8.0,Punatsangchhu,2023-12-30,73.889999,False,0.332977,268.139441,0.049192,-0.186830,-0.000250,...,0.000027,275.069172,0.000119,0.000022,2.250711e+06,0.000112,14,False,0.193197,0.204922


In [12]:
# === Build engineered features per basin_id ===
e = features.sort_values(["basin_id","date_local"]).reset_index(drop=True)

# Work on a copy we will extend
feat = e[["basin_id","date_local"]].copy()

# Flux windows (past-only rolling sums)
for v in flux_vars:
    if v in e.columns:
        for w in FLUX_WINDOWS:
            col = f"{v}_sum_{w}d"
            feat[col] = (e.groupby("basin_id")[v]
                           .apply(lambda s: strict_sum_past(s, w))
                           .reset_index(level=0, drop=True))

# State windows (past-only rolling means)
for v in state_vars:
    if v in e.columns:
        for w in STATE_WINDOWS:
            col = f"{v}_mean_{w}d"
            feat[col] = (e.groupby("basin_id")[v]
                           .apply(lambda s: strict_mean_past(s, w))
                           .reset_index(level=0, drop=True))

# API from precip (if present)
precip_col = next((c for c in e.columns if "precip" in c.lower()), None)
if precip_col:
    for h in API_HALFLIFE:
        col = f"api_d{h}"
        feat[col] = (e.groupby("basin_id")[precip_col]
                       .apply(lambda s: api_series(s, h))
                       .reset_index(level=0, drop=True))

# Calendar / seasonality
feat["month"]   = feat["date_local"].dt.month
doy             = feat["date_local"].dt.dayofyear
feat["doy_sin"] = np.sin(2*np.pi * (doy/365.25))
feat["doy_cos"] = np.cos(2*np.pi * (doy/365.25))
feat["is_monsoon"] = feat["month"].between(6,9).astype(int)

# Condition flags (freezing / snowpack) from original ERA5 columns
snow_depth_col = next((c for c in e.columns if "snow_depth" in c.lower()), None)
temp_col       = next((c for c in e.columns if c.lower().startswith(("t2m","temperature","2m_temperature"))), None)

if temp_col:
    T_c = to_celsius(e[temp_col])
    feat["is_freezing"] = (T_c <= 0).astype(int)
else:
    feat["is_freezing"] = 0

feat["has_snowpack"] = (e[snow_depth_col] > 0).astype(int) if snow_depth_col else 0
feat

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,vpd_kpa_mean_7d,vpd_kpa_mean_14d,api_d3,api_d7,month,doy_sin,doy_cos,is_monsoon,is_freezing,has_snowpack
0,3.0,2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,0.017202,0.999852,0,1,1
1,3.0,2000-01-02,-0.000222,NaN,NaN,NaN,NaN,0.000018,NaN,NaN,...,NaN,NaN,0.000018,0.000018,1,0.034398,0.999408,0,1,1
2,3.0,2000-01-03,-0.000248,NaN,NaN,NaN,NaN,0.000029,NaN,NaN,...,NaN,NaN,0.000043,0.000045,1,0.051584,0.998669,0,1,1
3,3.0,2000-01-04,-0.000225,-0.000695,NaN,NaN,NaN,0.000090,0.000136,NaN,...,NaN,NaN,0.000124,0.000130,1,0.068755,0.997634,0,1,1
4,3.0,2000-01-05,-0.000241,-0.000714,NaN,NaN,NaN,0.000015,0.000134,NaN,...,NaN,NaN,0.000113,0.000133,1,0.085906,0.996303,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29835,8.0,2023-12-27,-0.000247,-0.000764,-0.001802,-0.003084,-0.006859,0.000042,0.000191,0.000733,...,0.257938,0.219788,0.000707,0.002744,12,-0.073045,0.997329,0,0,1
29836,8.0,2023-12-28,-0.000223,-0.000748,-0.001751,-0.003146,-0.006743,0.000339,0.000403,0.001070,...,0.242584,0.227757,0.000901,0.002825,12,-0.055879,0.998438,0,0,1
29837,8.0,2023-12-29,-0.000246,-0.000716,-0.001713,-0.003203,-0.006698,0.000329,0.000711,0.001393,...,0.226444,0.229676,0.001044,0.002888,12,-0.038696,0.999251,0,0,1
29838,8.0,2023-12-30,-0.000276,-0.000745,-0.001719,-0.003253,-0.006682,0.000069,0.000737,0.001386,...,0.218724,0.230801,0.000898,0.002685,12,-0.021501,0.999769,0,0,1


## Join Static Basin Attributes

In [19]:
# Join static attributes
static = pd.read_parquet(STATIC_ATTR_PARQUET)

# Join on both basin_id and basin_name
final = feat.merge(
    static,
    on="basin_id",
    how="left",
    validate="m:1"  # ensures many-to-one merge: many daily rows per one basin
)

final.head()

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,Phaeozems_pct,Planosols_pct,Plinthosols_pct,Podzols_pct,Regosols_pct,Solonchaks_pct,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct
0,3.0,2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0
1,3.0,2000-01-02,-0.000222,NaN,NaN,NaN,NaN,0.000018,NaN,NaN,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,2000-01-03,-0.000248,NaN,NaN,NaN,NaN,0.000029,NaN,NaN,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,2000-01-04,-0.000225,-0.000695,NaN,NaN,NaN,0.000090,0.000136,NaN,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0
4,3.0,2000-01-05,-0.000241,-0.000714,NaN,NaN,NaN,0.000015,0.000134,NaN,...,0.0,0.0,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0


## Build Train (Exogenous-Only)

In [20]:
# === Join features ↔ target (exogenous-only) ===
train_exog = (feat_sorted
              .merge(y[["basin_id","date_local","discharge_cms","qc_any"]],
                     on=["basin_id","date_local"], how="left")
              .sort_values(["basin_id","date_local"])
              .reset_index(drop=True))

# Basic QA
n_all = len(train_exog)
n_miss_target = train_exog["discharge_cms"].isna().sum()
print(f"Train exogenous rows: {n_all} | missing target rows: {n_miss_target}")

# Write
train_exog.to_parquet(TRAIN_EXOG_PARQUET, index=False)
print("Wrote exogenous train:", TRAIN_EXOG_PARQUET)

NameError: name 'feat_sorted' is not defined

## Build Train (ARX: add lagged discharge)

In [21]:
if PRODUCE_ARX:
    te = train_exog.copy()

    # Add discharge lags & rolling stats per basin (leak-safe)
    te = te.sort_values(["basin_id","date_local"]).reset_index(drop=True)
    g = te.groupby("basin_id", group_keys=False)

    for L in ARX_LAGS:
        te[f"q_lag_{L}d"] = g["discharge_cms"].shift(L)

    for win, stat in ROLL_Q_STATS:
        if stat == "mean":
            te[f"q_roll{win}_mean"] = g["discharge_cms"].apply(lambda s: s.shift(1).rolling(win, min_periods=win).mean())
        elif stat == "std":
            te[f"q_roll{win}_std"]  = g["discharge_cms"].apply(lambda s: s.shift(1).rolling(win, min_periods=win).std())

    # Drop rows without full lag history or missing target
    needed = [f"q_lag_{L}d" for L in ARX_LAGS]
    before = len(te)
    te = te.dropna(subset=["discharge_cms"] + needed).reset_index(drop=True)
    after = len(te)
    print(f"ARX rows: {after} (dropped {before-after} due to lag history or missing target)")

    te.to_parquet(TRAIN_ARX_PARQUET, index=False)
    print("Wrote ARX train:", TRAIN_ARX_PARQUET)
else:
    print("PRODUCE_ARX=False → skipping ARX dataset.")


NameError: name 'train_exog' is not defined